# H3 Police Response Coverage

This notebook computes a complete H3 response-coverage grid for Western Australia.

The workflow:

1. Dense-polyfill every H3 cell that intersects the WA LGA boundary union at **resolution 7** (`h3.polygon_to_cells`, not a sparse lat/lon lattice).
2. Build a weighted, bidirectional road graph using segment length and configurable travel speeds.
3. Calculate shortest-road response times from every police station to each cell's nearest road segment.
4. Classify each cell by its fastest response time (unreachable cells stay **red**).
5. Export the complete coverage as GeoJSON (closed rings) and PMTiles (`-Z0 -z12`, no tiny-polygon drop).

> **Important:** The source road data does not contain a travel-speed field. The speed profile below is configurable and should be reviewed before operational use.


## Environment setup

Run the following command from the repository root before opening this notebook:

```bash
uv sync
uv run python -m jupyter notebook
```

The project already declares `h3`, `geopandas`, `shapely`, and `pyproj` through `pyproject.toml`. No additional Python dependencies are required for this workflow. The `tippecanoe` executable must be installed separately and should be available on the shell path.


In [ ]:
from __future__ import annotations

import heapq
import json
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path
from typing import Any

import geopandas as gpd
import h3
from h3 import LatLngPoly
from pyproj import Transformer
from shapely import STRtree
from shapely.geometry import Point, Polygon


REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "data" / "raw").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

RAW_DIR = REPOSITORY_ROOT / "data" / "raw"
PROCESSED_DIR = REPOSITORY_ROOT / "data" / "processed"
ROAD_PATH = RAW_DIR / "Road_Network.geojson"
STATION_PATH = RAW_DIR / "WAPoliceForceFacilities_SHP" / "Police_Facilities.shp"
BOUNDARY_PATH = RAW_DIR / "LGA_Boundaries_LGATE_233_WA_GDA2020_Public.geojson"

H3_RESOLUTION = 7
MAX_SAMPLE_DISTANCE_METRES = 5000.0
MAX_RESPONSE_MINUTES = 15.0

# Review these values before using the output operationally. The default profile
# is an illustrative estimate, not a measured policing response-time model.
ROAD_SPEED_KMH: dict[str, float] = {
    "State Road": 80.0,
    "Main Roads Controlled Path": 80.0,
    "Local Road": 50.0,
    "Miscellaneous Road": 40.0,
    "Proposed Road": 30.0,
    "Crossover": 60.0,
    "Unknown": 40.0,
}

RESPONSE_CLASSIFICATIONS = [
    {"label": "green", "min": 0.0, "max": 5.0, "color": "#2E7D32"},
    {"label": "yellow", "min": 5.0, "max": 10.0, "color": "#F9A825"},
    {"label": "orange", "min": 10.0, "max": 15.0, "color": "#EF6C00"},
    {"label": "red", "min": 15.0, "max": float("inf"), "color": "#C62828"},
]


def project_to_wgs84(frame: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if frame.crs is None:
        raise ValueError(f"{frame.crs} is missing. The source layer must declare a CRS.")
    if frame.crs.to_epsg() != 4326:
        return frame.to_crs(4326)
    return frame


def validate_inputs() -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, gpd.GeoDataFrame]:
    for path in (ROAD_PATH, STATION_PATH, BOUNDARY_PATH):
        if not path.exists():
            raise FileNotFoundError(f"Required input layer not found: {path}")

    roads = gpd.read_file(ROAD_PATH)
    stations = gpd.read_file(STATION_PATH)
    boundaries = gpd.read_file(BOUNDARY_PATH)
    roads = project_to_wgs84(roads)
    stations = project_to_wgs84(stations)
    boundaries = project_to_wgs84(boundaries)

    required_road_columns = {"START_NODE_NO", "END_NODE_NO", "NETWORK_TYPE", "geometry"}
    missing_road_columns = required_road_columns - set(roads.columns)
    if missing_road_columns:
        raise ValueError(f"Road layer is missing required columns: {sorted(missing_road_columns)}")

    roads = roads.copy()
    roads["NETWORK_TYPE"] = roads["NETWORK_TYPE"].fillna("Unknown").astype(str)
    roads["speed_kmh"] = roads["NETWORK_TYPE"].map(ROAD_SPEED_KMH).fillna(ROAD_SPEED_KMH["Unknown"])
    projected_roads = roads.to_crs(28350)
    projected_boundaries = boundaries.to_crs(28350)
    roads["valid_endpoint_pair"] = (
        roads["START_NODE_NO"].notna()
        & roads["END_NODE_NO"].notna()
        & roads.geometry.is_valid
        & projected_roads.geometry.length.gt(0)
        & roads["speed_kmh"].gt(0)
    )

    valid_roads = roads.loc[roads["valid_endpoint_pair"]].copy()
    valid_stations = stations.loc[stations.geometry.is_valid & stations.geometry.geom_type.eq("Point")].copy()
    valid_boundaries = boundaries.loc[boundaries.geometry.is_valid & projected_boundaries.geometry.area.gt(0)].copy()

    print(f"Road rows: {len(roads)}")
    print(f"Valid road rows: {len(valid_roads)}")
    print(f"Excluded road rows: {len(roads) - len(valid_roads)}")
    print(f"Station rows: {len(stations)}")
    print(f"Valid station rows: {len(valid_stations)}")
    print(f"Boundary rows: {len(boundaries)}")
    print(f"Valid boundary rows: {len(valid_boundaries)}")
    print(f"Road CRS: {roads.crs}")
    print(f"Station CRS: {stations.crs}")
    print(f"Boundary CRS: {boundaries.crs}")

    if valid_roads.empty or valid_stations.empty or valid_boundaries.empty:
        raise ValueError("At least one valid road, station, and boundary feature are required.")

    projected_roads = valid_roads.to_crs(28350)
    projected_stations = valid_stations.to_crs(28350)
    print(f"Projected road extent: {projected_roads.total_bounds.tolist()}")
    print(f"Projected station extent: {projected_stations.total_bounds.tolist()}")
    return valid_roads, valid_stations, valid_boundaries


In [ ]:
from shapely.ops import unary_union


def build_road_graph(roads: gpd.GeoDataFrame) -> tuple[dict[int, list[tuple[int, float]]], dict[int, int], list[dict[str, Any]]]:
    """Build a bidirectional weighted graph from road endpoint IDs."""
    nodes = set()
    for column in ("START_NODE_NO", "END_NODE_NO"):
        nodes.update(int(value) for value in roads[column].dropna().tolist())

    node_ids = {node: index for index, node in enumerate(sorted(nodes))}
    graph: dict[int, list[tuple[int, float]]] = defaultdict(list)
    segment_records: list[dict[str, Any]] = []

    projected_roads = roads.to_crs(28350)
    for row_index, row in projected_roads.iterrows():
        start_node = int(row["START_NODE_NO"])
        end_node = int(row["END_NODE_NO"])
        start_id = node_ids[start_node]
        end_id = node_ids[end_node]
        length_km = row.geometry.length / 1000.0
        speed_kmh = float(row["speed_kmh"])
        weight_minutes = length_km / speed_kmh * 60.0

        graph[start_id].append((end_id, weight_minutes))
        graph[end_id].append((start_id, weight_minutes))
        segment_records.append(
            {
                "row_index": row_index,
                "start_id": start_id,
                "end_id": end_id,
                "geometry": roads.loc[row_index].geometry,
                "projected_geometry": row.geometry,
                "network_type": str(row["NETWORK_TYPE"]),
                "speed_kmh": speed_kmh,
                "weight_minutes": weight_minutes,
            }
        )

    for node_id in graph:
        graph[node_id] = sorted(set(graph[node_id]))

    print(f"Graph nodes: {len(node_ids)}")
    print(f"Road segments: {len(segment_records)}")
    print(f"Graph edges: {sum(len(neighbors) for neighbors in graph.values()) // 2}")
    return dict(graph), node_ids, segment_records


def create_station_sources(
    stations: gpd.GeoDataFrame,
    road_tree: STRtree,
    road_segments: list[dict[str, Any]],
) -> list[tuple[int, int, float, float]]:
    """Connect each station to the nearest road segment and its endpoints."""
    sources: list[tuple[int, int, float, float]] = []

    for station_index, station in stations.iterrows():
        nearest_segment_index = int(road_tree.nearest(station.geometry))
        segment = road_segments[nearest_segment_index]
        station_point = station.geometry
        station_speed_kmh = segment["speed_kmh"]
        for endpoint_node in (segment["start_id"], segment["end_id"]):
            endpoint = (
                segment["projected_geometry"].coords[0]
                if endpoint_node == segment["start_id"]
                else segment["projected_geometry"].coords[-1]
            )
            direct_distance_km = station_point.distance(Point(endpoint)) / 1000.0
            if direct_distance_km <= 0:
                continue
            if direct_distance_km > MAX_SAMPLE_DISTANCE_METRES / 1000.0:
                continue
            sources.append(
                (
                    endpoint_node,
                    station_index,
                    max(0.01, direct_distance_km / station_speed_kmh * 60.0),
                    direct_distance_km,
                )
            )

    return sources


def ring_latlng(poly: Polygon) -> list[tuple[float, float]]:
    """Exterior ring as (lat, lng) for h3 LatLngPoly (drop closing vertex)."""
    coords = list(poly.exterior.coords)
    if coords[0] == coords[-1]:
        coords = coords[:-1]
    return [(lat, lng) for lng, lat in coords]


def polygon_to_cells_safe(poly: Polygon, res: int) -> set[str]:
    """Dense polyfill a Shapely polygon (holes as *variadic* LatLngPoly args)."""
    exterior = ring_latlng(poly)
    if len(exterior) < 3:
        return set()
    holes: list[list[tuple[float, float]]] = []
    for hole in poly.interiors:
        coords = list(hole.coords)
        if coords[0] == coords[-1]:
            coords = coords[:-1]
        hole_ring = [(lat, lng) for lng, lat in coords]
        if len(hole_ring) >= 3:
            holes.append(hole_ring)
    return set(h3.polygon_to_cells(LatLngPoly(exterior, *holes), res))


def generate_full_grid(boundaries: gpd.GeoDataFrame) -> list[str]:
    """Dense H3 polyfill of the WA boundary union via h3.polygon_to_cells."""
    boundary_geometry = unary_union(boundaries.geometry.tolist())
    cells: set[str] = set()

    if boundary_geometry.geom_type == "Polygon":
        polys = [boundary_geometry]
    elif boundary_geometry.geom_type == "MultiPolygon":
        polys = list(boundary_geometry.geoms)
    else:
        polys = [
            geom
            for geom in getattr(boundary_geometry, "geoms", [])
            if geom.geom_type == "Polygon"
        ]

    for poly in polys:
        cells.update(polygon_to_cells_safe(poly, H3_RESOLUTION))

    return sorted(cells)


def assign_h3_cells(
    segment_records: list[dict[str, Any]],
    full_grid: list[str],
) -> dict[str, list[int]]:
    """Assign each full-grid cell to its nearest projected road segment."""
    road_tree = STRtree([segment["projected_geometry"] for segment in segment_records])
    transformer = Transformer.from_crs(4326, 28350, always_xy=True)
    cells_to_segments: dict[str, list[int]] = {}

    for cell in full_grid:
        latitude, longitude = h3.cell_to_latlng(cell)
        cell_x, cell_y = transformer.transform(longitude, latitude)
        cell_point = Point(cell_x, cell_y)
        nearest_index = int(road_tree.nearest(cell_point))
        cells_to_segments[cell] = [nearest_index]

    return cells_to_segments


In [ ]:
def dijkstra(
    graph: dict[int, list[tuple[int, float]]],
    source: int,
    maximum_minutes: float | None = None,
) -> dict[int, float]:
    distances = {source: 0.0}
    queue: list[tuple[float, int]] = [(0.0, source)]

    while queue:
        current_distance, node = heapq.heappop(queue)
        if current_distance != distances.get(node):
            continue
        if maximum_minutes is not None and current_distance > maximum_minutes:
            break

        for neighbor, edge_minutes in graph.get(node, []):
            candidate = current_distance + edge_minutes
            if maximum_minutes is not None and candidate > maximum_minutes:
                continue
            if candidate < distances.get(neighbor, float("inf")):
                distances[neighbor] = candidate
                heapq.heappush(queue, (candidate, neighbor))

    return distances


def response_classification(response_minutes: float) -> str:
    if response_minutes < RESPONSE_CLASSIFICATIONS[0]["max"]:
        return RESPONSE_CLASSIFICATIONS[0]["label"]
    if response_minutes < RESPONSE_CLASSIFICATIONS[1]["max"]:
        return RESPONSE_CLASSIFICATIONS[1]["label"]
    if response_minutes < RESPONSE_CLASSIFICATIONS[2]["max"]:
        return RESPONSE_CLASSIFICATIONS[2]["label"]
    return RESPONSE_CLASSIFICATIONS[3]["label"]


def calculate_coverage(
    graph: dict[int, list[tuple[int, float]]],
    station_sources: list[tuple[int, int, float, float]],
    cells_to_segments: dict[str, list[int]],
    segment_records: list[dict[str, Any]],
    stations: gpd.GeoDataFrame,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    segment_endpoint_nodes = {
        segment_index: (segment["start_id"], segment["end_id"])
        for segment_index, segment in enumerate(segment_records)
    }
    station_indices = {station_index for _, station_index, _, _ in station_sources}
    cell_results: list[dict[str, Any]] = []
    station_times: dict[tuple[int, Any], list[float]] = defaultdict(list)

    for station_idx in sorted(station_indices):
        station_sources_for_station = [
            source for source in station_sources if source[1] == station_idx
        ]
        source_nodes = [source[0] for source in station_sources_for_station]
        source_access_times = [source[2] for source in station_sources_for_station]
        distances_by_source = [dijkstra(graph, source_node) for source_node in source_nodes]
        for cell, segment_indexes in cells_to_segments.items():
            endpoint_times = []
            for segment_index in segment_indexes:
                for endpoint_node in segment_endpoint_nodes[segment_index]:
                    for source_index, source_distances in enumerate(distances_by_source):
                        if endpoint_node in source_distances:
                            endpoint_times.append(
                                source_access_times[source_index] + source_distances[endpoint_node]
                            )
            if endpoint_times:
                station_times[(cell, station_idx)] = [min(endpoint_times)]

    for cell, segment_indexes in cells_to_segments.items():
        response_times = [
            station_times[(cell, station_index)]
            for station_index in sorted(station_indices)
            if (cell, station_index) in station_times
        ]
        if response_times:
            flattened_times = [value for group in response_times for value in group]
            min_response = min(flattened_times)
            mean_response = sum(flattened_times) / len(flattened_times)
            reachable_stations = len(flattened_times)
        else:
            # Keep every generated cell visible. Unreachable cells are shown as
            # red rather than disappearing from the map.
            min_response = MAX_RESPONSE_MINUTES + 0.01
            mean_response = min_response
            reachable_stations = 0
        cell_results.append(
            {
                "h3_cell": cell,
                "h3_resolution": H3_RESOLUTION,
                "response_class": response_classification(min_response),
                "min_response_minutes": round(min_response, 2),
                "mean_response_minutes": round(mean_response, 2),
                "reachable_stations": reachable_stations,
                "road_segment_count": len(segment_indexes),
                "source_layer": "Road_Network",
            }
        )

    metadata = {
        "h3_resolution": H3_RESOLUTION,
        "maximum_sample_distance_metres": MAX_SAMPLE_DISTANCE_METRES,
        "maximum_response_minutes": MAX_RESPONSE_MINUTES,
        "speed_profile_kmh": ROAD_SPEED_KMH,
        "classification_thresholds_minutes": {
            "green": "< 5",
            "yellow": "5-10",
            "orange": "10-15",
            "red": "> 15",
        },
        "method": (
            "Dense h3.polygon_to_cells polyfill of the WA LGA boundary union at resolution 7; "
            "shortest road-network travel time from each police station; "
            "response class uses the fastest reachable station."
        ),
        "station_to_road_method": "Nearest road segment, connected to both endpoints using the segment speed assumption.",
        "h3_fill_method": "h3.polygon_to_cells (LatLngPoly) on the boundary union -- not 0.1 deg lattice + grid_disk.",
        "limitations": [
            "Road speed values are estimates because source data has no speed field.",
            "Station-to-road connection uses the nearest segment and an endpoint-based estimate.",
            "Cells with no reachable station are retained as red and reported as unreachable within the maximum response window.",
        ],
    }
    return cell_results, metadata


In [ ]:
# The graph and H3 cell assignment are intentionally run separately so that
# failures are easy to diagnose before the potentially larger response run.
roads, stations, boundaries = validate_inputs()
projected_roads = roads.to_crs(28350)
projected_stations = stations.to_crs(28350)
road_tree = STRtree(projected_roads.geometry.tolist())
graph, node_ids, segment_records = build_road_graph(roads)
station_sources = create_station_sources(projected_stations, road_tree, segment_records)
full_grid = generate_full_grid(boundaries)
cells_to_segments = assign_h3_cells(segment_records, full_grid)

print(f"Full-grid H3 cells (dense polyfill): {len(full_grid)}")
print(f"Cells assigned to road segments: {len(cells_to_segments)}")
print(f"Station-to-road sources: {len(station_sources)}")


In [ ]:
cell_results, metadata = calculate_coverage(
    graph,
    station_sources,
    cells_to_segments,
    segment_records,
    stations,
)

print(f"Covered H3 cells: {len(cell_results)}")
for label in ("green", "yellow", "orange", "red"):
    count = sum(item["response_class"] == label for item in cell_results)
    print(f"{label}: {count}")


In [ ]:
def cell_ring_coordinates(cell: str) -> list[list[float]]:
    """Closed GeoJSON ring [lng, lat] for an H3 cell."""
    ring = [
        [longitude, latitude]
        for latitude, longitude in h3.cell_to_boundary(cell)
    ]
    if ring and ring[0] != ring[-1]:
        ring.append(ring[0])
    return ring


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_GEOJSON = PROCESSED_DIR / "Police_Response_Coverage.geojson"
OUTPUT_PMTILES = PROCESSED_DIR / "Police_Response_Coverage.pmtiles"
OUTPUT_METADATA = PROCESSED_DIR / "Police_Response_Coverage_metadata.json"

coverage_layer = {
    "type": "FeatureCollection",
    "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
    "features": [
        {
            "type": "Feature",
            "properties": {
                **result,
                "source_layer": "Road_Network",
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [cell_ring_coordinates(result["h3_cell"])],
            },
        }
        for result in cell_results
    ],
}

OUTPUT_GEOJSON.write_text(json.dumps(coverage_layer, indent=2), encoding="utf-8")
OUTPUT_METADATA.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Wrote GeoJSON: {OUTPUT_GEOJSON}")
print(f"Wrote metadata: {OUTPUT_METADATA}")

if shutil.which("tippecanoe") is None:
    raise RuntimeError("tippecanoe is not installed. Install it with your system package manager or ensure it is on PATH.")

# M4-validated flags: keep every hex through city zoom (no tiny-polygon attrition).
command = [
    "tippecanoe",
    "-o", str(OUTPUT_PMTILES),
    "-Z", "0",
    "-z", "12",
    "--force",
    "--no-tile-size-limit",
    "--no-feature-limit",
    "--no-tiny-polygon-reduction",
    "--no-tiny-polygon-reduction-at-maximum-zoom",
    "--drop-rate=0",
    "--detect-shared-borders",
    "-l", "Police_Response_Coverage",
    "-y", "h3_cell",
    "-y", "h3_resolution",
    "-y", "response_class",
    "-y", "min_response_minutes",
    "-y", "mean_response_minutes",
    "-y", "reachable_stations",
    "-y", "road_segment_count",
    "-y", "source_layer",
    str(OUTPUT_GEOJSON),
]
result = subprocess.run(command, capture_output=True, text=True, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Tippecanoe failed:\n{result.stdout}\n{result.stderr}")

print(f"Wrote PMTiles: {OUTPUT_PMTILES}")
print(f"PMTiles size: {OUTPUT_PMTILES.stat().st_size} bytes")


## Validation checklist

Before uploading the layer, verify:

- The GeoJSON is a dense H3 res-7 polyfill of the WA boundary (no lattice holes in Perth CBD).
- Polygon rings are closed; every cell has `h3_resolution`, a response class, and a finite response time (unreachable = red).
- Tippecanoe used `-Z0 -z12` with `--no-tiny-polygon-reduction` and `--drop-rate=0`.
- The metadata records the speed assumptions, dense-fill method, and limitations.
- The PMTiles file can be opened with the map client.
- The output is reviewed with an accurate speed profile before operational use.

The generated files are written to `data/processed` and are excluded from source control by the existing data ignore rules. S3 upload remains a separate, explicit step -- not part of this notebook run.
